# TuniScope — Campagne LoRA multitâche

Chaque graine produit un adaptateur Qwen partagé entre TSAC (sentiment) et MADAR (variétés arabes). En mode `full`, le notebook lance trois graines, deux passages sur les 4 000 exemples SFT, puis compare les résultats de validation à ceux de Qwen avant adaptation.

Sur Kaggle, les deux premières graines utilisent chacune une T4. La troisième prend la première carte disponible. Sur Mac, les essais se succèdent sur MPS. Le choix de la plateforme change la durée, tandis que le lot et le corpus de chaque essai restent fixes.

Le dépôt, le profil LoRA et les sources autorisées doivent être préparés. Sur Kaggle, coller la cellule de démarrage fournie séparément, au-dessus de ce notebook. Sur Mac, ouvrir depuis le dépôt. Le mode local par défaut est `smoke`, un contrôle de 100 exemples d’apprentissage par tâche suivi de petites vérifications d’inférence. Le mode `off` laisse parcourir les cellules. Le mode `full` lance la campagne scientifique.

In [ ]:
import json
import os
from pathlib import Path
import signal
import subprocess
import uuid

explicit = os.environ.get('TUNISCOPE_PROJECT_ROOT')
candidates = [Path(explicit)] if explicit else [Path.cwd(), *Path.cwd().parents]
ROOT = next((path.resolve() for path in candidates if (path / 'scripts/run_qwen_lora_campaign.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir depuis le dépôt à jour ou définir TUNISCOPE_PROJECT_ROOT.')

MODE = os.environ.get('TUNISCOPE_LORA_CAMPAIGN_MODE', 'smoke')  # smoke, full ou off
DEVICE = os.environ.get('TUNISCOPE_LORA_DEVICE', 'auto')
if MODE not in {'smoke', 'full', 'off'} or DEVICE not in {'auto', 'cpu', 'mps', 'cuda'}:
    raise ValueError('Vérifier le mode et l’appareil demandés.')

SFT = ROOT / 'outputs/qwen/sft_v1_2026-11-07'
EXISTING = os.environ.get('TUNISCOPE_LORA_CAMPAIGN_RESUME')
RUN = Path(EXISTING).resolve() if EXISTING else ROOT / 'outputs/qwen/notebooks' / f'{uuid.uuid4().hex}_{MODE}'
EXPORT = os.environ.get('TUNISCOPE_LORA_EXPORT_DIR')
if not EXPORT:
    EXPORT = str(ROOT / 'outputs/qwen/exports')

def run_script(script, *arguments):
    command = ['sh', str(ROOT / 'scripts/lora_uv.sh'), 'run', '--locked', '--offline',
               'python', '-u', str(ROOT / 'scripts' / script), *map(str, arguments)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except KeyboardInterrupt:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            process.wait()
            raise

print('Mode', MODE, '| appareil', DEVICE)
print('Dossier de travail', RUN)

## Lancer ou reprendre

Le contrôle SFT reproduit le corpus à partir des sources autorisées. En mode complet, chaque graine passe un contrôle court puis repart de la base initiale. Le dernier adaptateur est sauvegardé, rechargé et évalué.

Pour reprendre dans la même session, relancer la cellule de calcul. Pour reprendre depuis un autre noyau ayant accès au même dépôt temporaire, définir `TUNISCOPE_LORA_CAMPAIGN_RESUME` avec le dossier de travail affiché. Sur Kaggle, télécharger chaque ZIP depuis **Output** dès qu’il apparaît.

In [ ]:
if MODE != 'off':
    run_script('build_qwen_sft.py', '--verify' if SFT.exists() else '--output', SFT)
    options = ['--resume'] if EXISTING or (RUN / 'plan.json').is_file() else []
    run_script('run_qwen_lora_campaign.py', '--mode', MODE, '--device', DEVICE, '--dataset', SFT,
               '--output', RUN, '--export-dir', EXPORT, *options)

In [ ]:
if MODE == 'full':
    report = json.loads((RUN / 'campaign.json').read_text())
    print('Macro-F1 sur validation v2')
    print(f"{'Condition':20} {'Base':>7} {'13':>7} {'42':>7} {'2026':>7} {'Moyenne ± écart-type':>22}")
    for condition, row in report['validation'].items():
        scores = ' '.join(f"{row['macro_f1_by_seed'][str(seed)]:7.4f}" for seed in report['seeds'])
        print(f"{condition:20} {row['base_macro_f1']:7.4f} {scores} "
              f"{row['mean_macro_f1']:10.4f} ± {row['stdev_macro_f1']:.4f}")
if MODE != 'off':
    print('Dossier des archives par graine', EXPORT)